In [ ]:
import os
import pstats
import time
from pathlib import Path

import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
%load_ext snakeviz

## Configuration

In [ ]:
NUM_RUNS = int(os.getenv("NUM_RUNS", default=1))

In [ ]:
ECML2026_PKL = Path(os.getenv("ECML2026_PKL"))

In [ ]:
# download the ECML 2026 scenario used by benchmark_k_shortest_paths.py if not already present
if not ECML2026_PKL.exists():
    !wget "https://github.com/flatland-association/ecml2026-starterkit/raw/refs/heads/main/reinforcement_learning/sampling/level_0_scenario_1.pkl" -O {ECML2026_PKL}

# download the co-located station data benchmark_k_shortest_paths.py needs (Path(ECML2026_PKL).parent / "stations.pkl")
STATIONS_PKL = Path(os.getenv("ECML2026_STATIONS_PKL"))
if not STATIONS_PKL.exists():
    !wget "https://github.com/flatland-association/ecml2026-starterkit/raw/refs/heads/main/reinforcement_learning/sampling/stations.pkl" -O {STATIONS_PKL}

In [ ]:
loop = [
    {
        "sha": "5a97ccb6aec2e7c6227aba8a3b33de54f567ee3a",
        "date": "Tue Apr 23 15:17:36 2024 +0200",
        "name": "v4.0.2"
    },
    {
        "sha": "9115580bf7c602ca3c524ad392489bd712f355da",
        "date": "Tue Feb 18 17:03:18 2025 +0100",
        "name": "v4.0.4"
    },
    {
        "sha": "01d4c7ae8179c7a716059552eb31865772e5a549",
        "date": "Tue Feb 18 17:11:28 2025 +0100",
        "name": "118-fix-lru-cache-in-env-loading"
    },
    {
        "sha": "3f905a2bc37a0cd69047513d43df1576e7ba7634",
        "date": "Mon Mar 31 11:22:49 2025 +0200",
        "name": "179-simplify-step"
    },
    {
        "sha": "4fecd60e49dfb144b452f100ce916af2ed2a58fd",
        "date": "Mon Mar 31 18:16:23 2025 +0200",
        "name": "v4.1.0"
    },
    {
        "sha": "04911f88f50e30188b7d671291e0c2bbe1ee5ad1",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.1.1"
    },
    {
        "sha": "8f607149e29590a5baa5211d0efd32d1858091a3",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.0"
    },
    {
        "sha": "79c1031f4cb80e671fc6e58e9a590fd78865a4af",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.1"
    },
    {
        "sha": "d6a69ca00bde635f78b6b45032bdcfe6d2e480aa",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.2"
    },
    {
        "sha": "189c259a8fe19266329a6826a137e1db29a12996",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.3"
    },
    {
        "sha": "dfcea58bfed32c9b549fba04a86e57a532280dc2",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.4"
    },
    {
        "sha": "2156f46ffd8c707c8737fe00da89376f9cc5c4e4",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "pr-402"
    },
    {
        "sha": "6de179e336d6d7fe40a385a90ca9d1ac328fc786",
        "date": "Fri May 16 16:57:14 2025 +0200",
        "name": "v4.2.5"
    },
    {
        "sha": "71e5faa53fae3a67b4dc35e504018701e1d3fe83",
        "date": "Tue Jun 2 09:23:04 2026 +0200",
        "name": "v4.2.6"
    },
    {
        "sha": "LOCAL",
        "date": "--",
        "name": "LOCAL"
    },
    {
        "sha": "LOCAL_Cython",
        "date": "--",
        "name": "LOCAL_Cython"
    }
]

## Run Benchmarks with Profiler

In [ ]:
# use separate location where we checkout the code version to profile
# hence we can control the configuration of the env in the cli in the current code relative to this notebook
# CAVEAT: we run the checked out code in the current env (as we run it from the notebook) - this could lead to inconsistencies in the future (discarded or updated requirements or backwards-incompatibilities of the benchmarking cli)
!git clone https://github.com/flatland-association/flatland-rl.git /tmp/flatland-rl
!cd /tmp/flatland-rl && git clean -fdx && git reset --hard

In [ ]:
for i in range(NUM_RUNS):
    for l in loop:
        if l["name"] in ("LOCAL", "LOCAL_Cython"):
            continue
        print("===================================================================")
        print(f'{l["name"]} - {l["sha"]} - {i}')
        print("===================================================================")
        !cd /tmp/flatland-rl && git checkout {l["sha"]} && git log -1
        !export PYTHONPATH=/tmp/flatland-rl && export ECML2026_PKL={ECML2026_PKL} && python -m cProfile -o benchmark_k_shortest_paths.py_{l["name"]}_{i}.prof -m pytest benchmark_k_shortest_paths.py
        time.sleep(2)

In [ ]:
# ensure no compiled Cython artifacts remain so LOCAL (and all other non-Cython loop configs) reflect pure-Python performance
repo_root = Path.cwd().parent
!rm -rf {repo_root}/build {repo_root}/flatland/envs/step_utils/*.c {repo_root}/flatland/envs/step_utils/*.so

In [ ]:
for i in range(NUM_RUNS):
    print("===================================================================")
    print(f'LOCAL - {i}')
    print("===================================================================")
    !export PYTHONPATH={Path.cwd().parent} && export ECML2026_PKL={ECML2026_PKL} && python -m cProfile -o benchmark_k_shortest_paths.py_LOCAL_{i}.prof -m pytest benchmark_k_shortest_paths.py

### Cythonize `ext_modules` and run LOCAL_Cython

In [ ]:
# cythonize + compile the ext_modules declared in pyproject.toml's [tool.setuptools] ext-modules (in-place,
# so the .so lands next to the .py it shadows) - there is no setup.py anymore, so invoke setup() directly
!python -m pip install "cython>=3.3.0a1" "setuptools_scm>=8"
!cd {repo_root} && python -c "from setuptools import setup; setup()" build_ext --inplace

In [ ]:
for i in range(NUM_RUNS):
    print("===================================================================")
    print(f'LOCAL_Cython - {i}')
    print("===================================================================")
    !export PYTHONPATH={repo_root} && export ECML2026_PKL={ECML2026_PKL} && python -m cProfile -o benchmark_k_shortest_paths.py_LOCAL_Cython_{i}.prof -m pytest benchmark_k_shortest_paths.py

In [ ]:
# restore the working tree to a pure-Python state so no other loop config or downstream test picks up the compiled extension
!rm -rf {repo_root}/build {repo_root}/flatland/envs/step_utils/*.c {repo_root}/flatland/envs/step_utils/*.so

## Analyse Profiling

In [ ]:
# https://stackoverflow.com/questions/44302726/pandas-how-to-store-cprofile-output-in-a-pandas-dataframe
def prof_to_df(st):
    keys_from_k = ['file', 'line', 'fn']
    keys_from_v = ['cc', 'ncalls', 'tottime', 'cumtime', 'callers']
    data = {k: [] for k in keys_from_k + keys_from_v}

    s = st.stats

    for k in s.keys():
        for i, kk in enumerate(keys_from_k):
            data[kk].append(k[i])

        for i, kk in enumerate(keys_from_v):
            data[kk].append(s[k][i])
    return pd.DataFrame(data)

In [ ]:
agg = {"fn": ["first"], "sha": ["first"], "cumtime": ['mean', 'median', 'min', 'max', 'std'], "tottime": ['mean', 'median', 'min', 'max', 'std']}

In [ ]:
def aggregate(example):
    dfs = []
    for l in loop:
        for i in range(NUM_RUNS):
            fn = f'{example}_{l["name"]}_{i}.prof'
            ps = pstats.Stats(fn)
            # print(fn)
            ps = pstats.Stats(fn)
            df = prof_to_df(ps)
            df["sha"] = l["sha"]
            df["name"] = l["name"]
            dfs.append(df)
    df = pd.concat(dfs)
    return df

In [ ]:
# tottime is the total time spent in the function alone.
# cumtime is the total time spent in the function plus all functions that this function called.

In [ ]:
def filter_df(df, conditions):
    cond = False
    for fn, file in conditions:
        cond = cond | (df["fn"] == fn) & (df["file"].str.contains(file))
    return df[cond]

In [ ]:
def analyse_df(df, fn, file, sort_by="cumtime"):
    df_ = df[(df["fn"] == fn) & (df["file"].str.contains(file))].groupby("name").agg(agg).sort_values((sort_by, "median"), ascending=True)
    df_["diff_median"] = df_[(sort_by, "median")].diff().cumsum()
    df_["diff%_median"] = df_["diff_median"] / (df_[("cumtime", "median")] + df_["diff_median"]) * 100
    df_["diff_mean"] = df_[(sort_by, "mean")].diff().cumsum()
    df_["diff%_mean"] = df_["diff_mean"] / (df_[("cumtime", "mean")] + df_["diff_mean"]) * 100
    return df_

In [ ]:
df_benchmark_k_shortest_paths = aggregate("benchmark_k_shortest_paths.py")
df_benchmark_k_shortest_paths

### Look into overall performance

In [ ]:
plt.figure(figsize=(15, 8))
ax = sns.barplot(filter_df(df_benchmark_k_shortest_paths, [
    ("get_k_shortest_paths", "rail_env_shortest_paths.py"),
]), x="name", y="cumtime", hue="fn", legend=True, estimator="median")
ax.bar_label(ax.containers[0], fontsize=10);
plt.savefig("performance_get_k_shortest_paths.png")

The same data in tabular form:

In [ ]:
analyse_df(df_benchmark_k_shortest_paths, "get_k_shortest_paths", "rail_env_shortest_paths.py")

### Snakeviz of individual profiles
Use the following line to start a snakeviz server and open a new browser window:

In [ ]:
#!snakeviz "benchmark_k_shortest_paths.py_LOCAL_3.prof"